# INT8 MatMul with Fused Dequant

Bot task id: `12_matmul_int8_fused_dequant`, deadline 06.12.2026 00:00 MSK. [Open in Colab](https://colab.research.google.com/github/PE51K/itmo-ai-talent-hub-efficient-models-course/blob/main/hw3/12_matmul_int8_fused_dequant.ipynb)

## Условие

INT8 матмул с фьюзом деквантизации и биаса. Даны:
- `X_q` формы `(B, IN)` — int8 активации (row-major, contiguous);
- `W_q` формы `(IN, OUT)` — int8 веса (row-major, contiguous);
- `s_x` — fp32 скаляр масштаба для `X_q` (per-tensor);
- `s_w` — fp32 масштабы для `W_q`: скаляр (per-tensor) или вектор `(OUT,)` (per-OUT-channel);
- `bias` — опционально fp16 вектор `(OUT,)`.

Выход `Y` формы `(B, OUT)` в fp16:
1. накапливайте int8·int8 в int32: $ACC[b, o] = \sum_k X_q[b, k] \cdot W_q[k, o]$;
2. масштаб на столбец: $\alpha[o] = s_x \cdot s_w[o]$ (per-channel) или $s_x \cdot s_w$;
3. после K-цикла: $Y[b, o] = \text{fp16}(\text{fp32}(ACC[b, o]) \cdot \alpha[o] + bias[o])$.

**Требования**
- ядро грузит тайлы int8 из `X_q` и `W_q`, аккумулирует в int32, применяет масштабы один раз после K-цикла, добавляет bias внутри ядра (если есть), пишет `Y` в fp16 (row-major);
- per-channel и per-tensor масштабы через constexpr-флаг `PER_CHANNEL`;
- маскировать хвосты по всем измерениям;
- `tl.load` и `tl.store`;
- `@triton.autotune` с не менее чем 3 конфигурациями.

`b_ptr` есть в ядре всегда; когда `bias=None`, нужен способ его не читать (например, ещё один constexpr-флаг).

**T4.** Работает ли int8 `tl.dot` на T4 (sm_75), не проверено. Если Triton откажется компилировать, ошибка будет в выводе проверки.

## Setup

Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import importlib.util

import torch
import triton

assert torch.cuda.is_available(), "no GPU in this runtime: Runtime -> Change runtime type -> T4 GPU"
print(f"torch {torch.__version__}, triton {triton.__version__}, {torch.cuda.get_device_name()}")

## Solution

`%%writefile` saves the cell to `12_matmul_int8_fused_dequant.py`, the file to submit. Run it after every edit, then the check.

In [ ]:
%%writefile 12_matmul_int8_fused_dequant.py
import torch
import triton
import triton.language as tl


@triton.autotune(
    configs=[
        triton.Config({"BLOCK_M": 64, "BLOCK_N": 64, "BLOCK_K": 64}, num_warps=4, num_stages=2),
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 64, "BLOCK_K": 64}, num_warps=4, num_stages=2),
        triton.Config({"BLOCK_M": 64, "BLOCK_N": 128, "BLOCK_K": 64}, num_warps=4, num_stages=2),
    ],
    key=["B", "IN", "OUT"],
)
@triton.jit
def _forward_int8_fused_kernel(x_q_ptr, x_scale_ptr,
                               w_q_ptr, w_scale_ptr,
                               b_ptr, y_ptr,
                               B, IN, OUT,
                               BLOCK_M: tl.constexpr,
                               BLOCK_N: tl.constexpr,
                               BLOCK_K: tl.constexpr,
                               PER_CHANNEL: tl.constexpr):
    # YOUR CODE HERE
    pass


def matmul_int8_fused(x_q: torch.Tensor,
                      x_scale: torch.Tensor,
                      w_q: torch.Tensor,
                      w_scale: torch.Tensor,
                      bias: torch.Tensor | None = None,
                      *, per_channel: bool = True) -> torch.Tensor:
    """Вернуть Y = dequant(X_q) @ dequant(W_q) + bias, dtype fp16, shape (B, OUT)."""
    # YOUR CODE HERE
    raise NotImplementedError

## Check

Compares with PyTorch on shapes with tails and checks some of the requirements. These are my tests, not the bot's: passing them doesn't guarantee the bot accepts the solution.

In [ ]:
PATH = "12_matmul_int8_fused_dequant.py"
spec = importlib.util.spec_from_file_location("sol", PATH)
sol = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sol)
SRC = open(PATH).read()

B, IN, OUT = 77, 300, 130
x_q = torch.randint(-127, 128, (B, IN), device="cuda", dtype=torch.int8)
w_q = torch.randint(-127, 128, (IN, OUT), device="cuda", dtype=torch.int8)
s_x = torch.tensor([0.01], device="cuda")
bias = torch.randn(OUT, device="cuda", dtype=torch.half)
acc = x_q.float() @ w_q.float()
for per_channel, s_w in ((True, torch.rand(OUT, device="cuda") * 1e-3), (False, torch.tensor([1e-3], device="cuda"))):
    for b in (bias, None):
        ref = acc * (s_x * s_w) + (b.float() if b is not None else 0)
        y = sol.matmul_int8_fused(x_q, s_x, w_q, s_w, b, per_channel=per_channel)
        assert y.dtype == torch.float16 and y.shape == (B, OUT), (y.dtype, y.shape)
        torch.testing.assert_close(y, ref.half(), atol=1e-2, rtol=1e-2)
assert len(sol._forward_int8_fused_kernel.configs) >= 3, "the task asks for at least 3 autotune configs"

torch.cuda.synchronize()
print("PASS")

## Download

In [ ]:
from google.colab import files

files.download("12_matmul_int8_fused_dequant.py")